<a href="https://colab.research.google.com/github/EsarFatima/MachineLearning-flyrank-/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EsarFatima/MachineLearning-flyrank-"
REPO_DIR = "MachineLearning-flyrank-"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])

num_features = ["content_age_days", "days_since_last_update", "log_impressions_90d",
                 "avg_position", "ctr", "engagement_rate", "search_volume", "competition",
                 "word_count", "has_keyword_data", "has_word_count"]
cat_features = ["content_type", "main_intent"]

X_num = df[num_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_cat = pd.get_dummies(df[cat_features].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1)
y = df["is_declining_label"]
groups = df["client_id"]

# Same honest, client-grouped split as w06 -- the model below is fit ONLY on the 25 training
# clients, then used to SCORE every row (including the 7 it never trained on and never will).
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                             class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

df["rf_score"] = rf.predict_proba(X)[:, 1]
df["seen_in_training"] = df.index.isin(train_idx)

# Reason codes: combine the w04 rule (stale + visible) with model risk tiers cut at THIS
# model's own p50/p90, since its raw scores are compressed (0.01-0.74), not spread 0-1.
p50, p90 = df["rf_score"].quantile(0.50), df["rf_score"].quantile(0.90)
stale = df["days_since_last_update"] >= 90
visible = (df["impressions_90d"] >= 300) & (df["impressions_90d"] < 30000)
rule_flag = stale & visible
df["risk_tier"] = pd.cut(df["rf_score"], bins=[-1, p50, p90, 2], labels=["low", "moderate", "high"])

df["reason_code"] = "low_priority"
df["action"] = "no_action"
both_high = rule_flag & (df["risk_tier"] == "high")
model_only = (~rule_flag) & (df["risk_tier"] == "high")
rule_mod = rule_flag & (df["risk_tier"] == "moderate")
rule_low_model = rule_flag & (df["risk_tier"] == "low")
df.loc[both_high, ["reason_code", "action"]] = ["stale_visible_and_top_risk_decile", "review_for_refresh_priority"]
df.loc[model_only, ["reason_code", "action"]] = ["top_risk_decile_not_flagged_by_rule", "review_for_refresh_secondary"]
df.loc[rule_mod, ["reason_code", "action"]] = ["stale_visible_moderate_model_risk", "review_for_refresh_secondary"]
df.loc[rule_low_model, ["reason_code", "action"]] = ["stale_visible_but_low_model_risk", "spot_check_only"]

print(df["reason_code"].value_counts())
print()
print(df["action"].value_counts())

reason_code
low_priority                           21753
stale_visible_but_low_model_risk        2668
stale_visible_moderate_model_risk       2579
top_risk_decile_not_flagged_by_rule     1502
stale_visible_and_top_risk_decile       1498
Name: count, dtype: int64

action
no_action                       21753
review_for_refresh_secondary     4081
spot_check_only                  2668
review_for_refresh_priority      1498
Name: count, dtype: int64


In [2]:
# Hand-review the top of the list, per the skill card: this is where bad logic shows itself.
queue_cols = ["content_id", "client_id", "rf_score", "risk_tier", "reason_code", "action",
              "seen_in_training", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
queue = df[queue_cols].sort_values("rf_score", ascending=False).reset_index(drop=True)

for k in [20, 50, 100, 200]:
    top = queue.head(k)
    print(f"top {k}: {top['client_id'].nunique()} unique client(s), "
          f"top client = {top['client_id'].value_counts().iloc[0]/k:.0%} of the list")

top 20: 1 unique client(s), top client = 100% of the list
top 50: 1 unique client(s), top client = 100% of the list
top 100: 2 unique client(s), top client = 99% of the list
top 200: 3 unique client(s), top client = 96% of the list


**The weak pick the hand review is supposed to find:** the top 20, top 50, and 99% of the top 100 belong to a single client, `client_3fdba35f04`. Checking why — that client's own decline rate is 83.9% versus 51.8% for everyone else, and its median `days_since_last_update` is 104 days versus 20 days portfolio-wide. So this isn't the model malfunctioning; it's one genuinely, uniformly stale client's content getting ranked together at the top of a *pooled, cross-client* queue. That's still a real limitation for how this queue gets used (Section 2) — a global editor reading straight down the list would spend their first day entirely on one account. The reason codes are honest about *why* each row is there; they don't protect against one client swamping the top of a shared queue, which is a human's job to catch before assigning work.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [3]:
print("Rows from clients the model NEVER trained on:", (~df["seen_in_training"]).sum(),
      "of", len(df), f"({(~df['seen_in_training']).mean():.0%})")
print("Validated precision@50 (7 held-out clients, from w06):  0.56  (base rate 0.511, rule 0.38)")
print("Validated precision@50 for the other 25 (seen) clients: NOT independently validated --")
print("their rows contributed to training, so any precision computed on them would be optimistic.")

Rows from clients the model NEVER trained on: 6163 of 30000 (21%)
Validated precision@50 (7 held-out clients, from w06):  0.56  (base rate 0.511, rule 0.38)
Validated precision@50 for the other 25 (seen) clients: NOT independently validated --
their rows contributed to training, so any precision computed on them would be optimistic.


**Intended user:** a content strategist or SEO editor deciding which pages in an active portfolio to review for a refresh this cycle — a triage aid for prioritizing attention, not a publishing or auto-refresh trigger.

**What it's validated for:** ranking pages by decline risk *within a client the model has already seen labeled outcomes for*, at the precision@50 level measured in w06 (0.56 vs. a 0.511 base rate). That number came from 7 clients the model never trained on, which is the closest proxy we have to "a brand-new client tomorrow" — but 7 is a small holdout, so I'd treat 0.56 as a reasonable estimate, not a guarantee.

**Where it stops being valid:**
- **A genuinely new client with no historical labels at all.** Every feature here (CTR, engagement, position, staleness) needs 90 days of GSC/GA4 history to mean anything; a brand-new account has none of that yet, and the queue shouldn't be trusted for it until it accumulates a comparable window.
- **Single-client pileups**, per Section 1 — the queue ranks pages, not clients, so a client with a uniformly bad staleness problem will dominate a shared queue's top even though every one of its individual pages is only moderately differentiated from its siblings.
- **Content outside this portfolio's shape** — the training data is 30K rows from one internship warehouse snapshot; a different content type, industry, or market with a different baseline CTR or seasonality pattern hasn't been tested here at all.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**A person must check, before any page moves from this queue into someone's work:**
- Whether the top of today's pull is dominated by one client (Section 1's finding) — if so, cap how many rows from a single client can occupy the front of the queue, or review clients on a rotating basis instead of reading straight down the ranked list.
- Whether a flagged page's `ctr == 0` or `engagement_rate == 0` — per w04/w06's own findings, zero-click and zero-engagement pages are a different population (often too new or under-measured for the signal to mean anything), not simply "the worst" pages, and can produce a reason code that reads more confidently than the evidence behind it.
- Whether the page belongs to one of the 21% of rows from a held-out client versus the 79% the model already saw labeled outcomes for (Section 2) — scores for seen clients carry less independent validation than the headline precision@50 number implies.

**Never automated — always a human decision:**
- Actually rewriting, unpublishing, or redirecting a page. This queue only says *review*; it never says *change*, and it has no signal at all about what a rewrite should say.
- Any action on `top_risk_decile_not_flagged_by_rule` rows without a human sanity-check first — this is exactly the group where the model and the transparent w04 rule disagree, so a person needs to read the actual page and confirm the signal makes sense before treating it as equal-priority to a rule-and-model agreement.
- Treating `stale_visible_but_low_model_risk` (the `spot_check_only` rows, where the rule flags but the model doesn't) as "safe to ignore" — it means the two disagree, not that the rule is wrong; it stays a spot-check, never a silent drop.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
# A concrete trigger this data can already compute: the score distribution itself.
# If a future month's scores drift far from this baseline, that's a retrain signal, not
# something to interpret as "content suddenly got riskier."
print("Baseline rf_score distribution (this snapshot, for future drift comparison):")
print(df["rf_score"].describe(percentiles=[.1, .5, .9]).round(3).to_dict())
print()
print("Baseline reason-code mix (this snapshot):")
print((df["reason_code"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")

Baseline rf_score distribution (this snapshot, for future drift comparison):
{'count': 30000.0, 'mean': 0.507, 'std': 0.167, 'min': 0.011, '10%': 0.27, '50%': 0.556, '90%': 0.675, 'max': 0.742}

Baseline reason-code mix (this snapshot):
reason_code
low_priority                           72.5%
stale_visible_but_low_model_risk        8.9%
stale_visible_moderate_model_risk       8.6%
top_risk_decile_not_flagged_by_rule     5.0%
stale_visible_and_top_risk_decile       5.0%
Name: proportion, dtype: object


**Retrain/re-review triggers:**
- **Reason-code mix drift.** If a future pull's `low_priority` share drops meaningfully below ~72%, or `top_risk_decile_not_flagged_by_rule` grows well past ~5%, that's a sign the portfolio's underlying feature distributions have shifted enough that the p50/p90 cut points (computed on this snapshot) no longer describe "typical" vs. "risky" — recompute the tiers, don't keep the old thresholds.
- **New clients accumulating history.** Once a client crosses roughly 90 days of tracked impressions and sessions (this dataset's own floor for a meaningful signal), re-score it rather than leaving it excluded or scored on thin data.
- **A scheduled full retrain**, not just a re-score — at minimum whenever the sealed test month (`month=2026-06`) is unsealed for a new evaluation cycle, so the model gets refreshed labels instead of re-running forever on the same fit.
- **Any single client re-appearing at the top of the queue for multiple consecutive cycles** — per Section 1, that's either a real, ongoing staleness problem worth escalating directly to that account, or a sign the pooled ranking needs a per-client normalization step added.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/content_action_queue.csv", index=False)

summary = pd.DataFrame([
    {"metric": "precision@50, held-out clients", "value": 0.56, "note": "base rate 0.511, w04 rule 0.38"},
    {"metric": "AUC, held-out clients (RF)", "value": 0.591, "note": "vs 0.723 on a leaky random split"},
    {"metric": "rows flagged review_for_refresh_priority", "value": int((queue["action"] == "review_for_refresh_priority").sum()),
     "note": f"of {len(queue)} total"},
    {"metric": "share of queue from held-out clients", "value": round((~df['seen_in_training']).mean(), 3),
     "note": "the only independently-validated slice"},
])
summary.to_csv("work/outputs/playbook_summary_metrics.csv", index=False)

print("Wrote:")
print(" - work/outputs/content_action_queue.csv   (", len(queue), "rows )")
print(" - work/outputs/playbook_summary_metrics.csv")
print()
print(summary.to_string(index=False))

Wrote:
 - work/outputs/content_action_queue.csv   ( 30000 rows )
 - work/outputs/playbook_summary_metrics.csv

                                  metric    value                                   note
          precision@50, held-out clients    0.560         base rate 0.511, w04 rule 0.38
              AUC, held-out clients (RF)    0.591       vs 0.723 on a leaky random split
rows flagged review_for_refresh_priority 1498.000                         of 30000 total
    share of queue from held-out clients    0.205 the only independently-validated slice


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.